In [1]:
from scipy.stats import chi2_contingency
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('../data/Telc_customer_churn_cleaned.csv')
df

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Churn Reason,Contract_One year,Contract_Two year,Internet Service_Fiber optic,Internet Service_No,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check,avg_monthly_spend,has_multiple_services
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,1,...,Competitor made better offer,False,False,False,False,False,False,True,54.075000,True
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,0,...,Moved,False,False,True,False,False,True,False,75.825000,False
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,0,...,Moved,False,False,True,False,False,True,False,102.562500,False
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,0,...,Moved,False,False,True,False,False,True,False,108.787500,False
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,1,...,Competitor had better devices,False,False,True,False,False,False,False,102.781633,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7027,2569-WGERO,1,United States,California,Landers,92285,"34.341737, -116.539416",34.341737,-116.539416,0,...,NaN,False,True,False,True,False,False,False,19.713889,False
7028,6840-RESVB,1,United States,California,Adelanto,92301,"34.667815, -117.536183",34.667815,-117.536183,1,...,NaN,True,False,False,False,False,False,True,82.937500,True
7029,2234-XADUH,1,United States,California,Amboy,92304,"34.559882, -115.637164",34.559882,-115.637164,0,...,NaN,True,False,True,False,True,False,False,102.262500,False
7030,4801-JZAZL,1,United States,California,Angelus Oaks,92305,"34.1678, -116.86433",34.167800,-116.864330,0,...,NaN,False,False,False,False,False,True,False,31.495455,False


In [4]:
df_stayed = df[df['Churn Value'] == 0]
df_churned = df[df['Churn Value'] == 1]

In [23]:
table = pd.crosstab(index=df['Churn Value'], columns=df['Tech Support'])
chi2, p, dof, expected = chi2_contingency(table)
print(dof)

2


### For each chi squared test, churn value would be the rows and categorial data would constitute the columns

In [28]:
from statsmodels.stats.multitest import multipletests

service_cols = ['Multiple Lines', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Paperless Billing']

rows = []

# Tests for Additional Services vs Churn

def chi2_tests():

    for col in service_cols:
        table = pd.crosstab(index=df['Churn Value'], columns=df[col])
        chi2, p, dof, expected = chi2_contingency(table)
        n = table.to_numpy().sum()
        r, c = table.shape
        k_minus_1 = min(r - 1, c - 1)
        cramers_v = (chi2/(n*k_minus_1))**0.5
        rows.append({'feature': col, 'chi2': chi2, 'p': p, 'cramers_v': cramers_v})

chi2_tests()

results = pd.DataFrame(rows)
results['p_holm'] = multipletests(results['p'], method='holm')[1]
results.sort_values('cramers_v', ascending=False)
print(results)



             feature        chi2              p  cramers_v         p_holm
0     Multiple Lines   11.271541   3.567927e-03   0.040036   3.567927e-03
1    Online Security  846.677389  1.400687e-184   0.346992  1.120549e-183
2      Online Backup  599.175185  7.776099e-131   0.291902  4.665660e-130
3  Device Protection  555.880327  1.959389e-121   0.281159  9.796944e-121
4       Tech Support  824.925564  7.407808e-180   0.342506  5.185465e-179
5       Streaming TV  372.456502   1.324641e-81   0.230143   3.973923e-81
6   Streaming Movies  374.268432   5.353560e-82   0.230702   2.141424e-81
7  Paperless Billing  256.874908   8.236203e-58   0.191127   1.647241e-57


<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str